Data Enrichment

Read the concatinated data
Handle Missing values
Create derived columns for further analysis
Save the data in data/JC

Loading Libraries

In [1]:
import requests

import numpy as np
import pandas as pd

import plotly.express as px
import folium

import os
from pathlib import Path
from urllib.request import urlretrieve
from urllib.error import HTTPError, URLError
from zipfile import ZipFile

In [2]:
citibike_df = pd.read_csv("../data/JC/JC2025.csv")

In [3]:
citibike_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1002704 entries, 0 to 1002703
Data columns (total 13 columns):
 #   Column              Non-Null Count    Dtype  
---  ------              --------------    -----  
 0   ride_id             1002704 non-null  str    
 1   rideable_type       1002704 non-null  str    
 2   started_at          1002704 non-null  str    
 3   ended_at            1002704 non-null  str    
 4   start_station_name  1002701 non-null  str    
 5   start_station_id    1002701 non-null  str    
 6   end_station_name    999469 non-null   str    
 7   end_station_id      998307 non-null   str    
 8   start_lat           1002702 non-null  float64
 9   start_lng           1002702 non-null  float64
 10  end_lat             999260 non-null   float64
 11  end_lng             999260 non-null   float64
 12  member_casual       1002704 non-null  str    
dtypes: float64(4), str(9)
memory usage: 99.5 MB


In [4]:
citibike_df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,9F734BE1BFC45FF4,electric_bike,2025-11-18 18:34:14.943,2025-11-18 18:47:33.391,Glenwood Ave,JC094,West Side Ave & Stegman Pkwy,JC131,40.727551,-74.071061,40.710870,-74.093680,member
1,B6C773B13AC0E465,classic_bike,2025-11-26 16:29:15.513,2025-11-26 16:43:45.235,Glenwood Ave,JC094,West Side Ave & Stegman Pkwy,JC131,40.727551,-74.071061,40.710870,-74.093680,member
2,C300465AA158280F,electric_bike,2025-11-04 22:31:58.010,2025-11-04 22:38:57.017,Bloomfield St & 15 St,HB203,Marshall St & 2 St,HB408,40.754530,-74.026580,40.740802,-74.042521,member
3,31A424FC97C8AAFB,classic_bike,2025-11-08 06:51:57.424,2025-11-08 06:57:45.627,Clinton St & 7 St,HB303,Marshall St & 2 St,HB408,40.745420,-74.033320,40.740802,-74.042521,member
4,08C5EA04CB1FDC57,classic_bike,2025-11-24 20:31:21.758,2025-11-24 20:38:01.261,Clinton St & 7 St,HB303,Marshall St & 2 St,HB408,40.745420,-74.033320,40.740802,-74.042521,member


Observe Missing Values

In [5]:
missing_values = (
    citibike_df
    .isna()
    .sum()
    .reset_index()
)

missing_values.columns = ["column", "missing_count"]

missing_values["missing_percentage"] = (
    missing_values["missing_count"] / len(citibike_df)
)*100

missing_values.sort_values("missing_count", ascending=False)

,column,missing_count,missing_percentage
7,end_station_id,4397,0.438514
10,end_lat,3444,0.343471
11,end_lng,3444,0.343471
6,end_station_name,3235,0.322628
4,start_station_name,3,0.000299
5,start_station_id,3,0.000299
8,start_lat,2,0.000199
9,start_lng,2,0.000199
0,ride_id,0,0.000000
1,rideable_type,0,0.000000


In [6]:
citibike_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1002704 entries, 0 to 1002703
Data columns (total 13 columns):
 #   Column              Non-Null Count    Dtype  
---  ------              --------------    -----  
 0   ride_id             1002704 non-null  str    
 1   rideable_type       1002704 non-null  str    
 2   started_at          1002704 non-null  str    
 3   ended_at            1002704 non-null  str    
 4   start_station_name  1002701 non-null  str    
 5   start_station_id    1002701 non-null  str    
 6   end_station_name    999469 non-null   str    
 7   end_station_id      998307 non-null   str    
 8   start_lat           1002702 non-null  float64
 9   start_lng           1002702 non-null  float64
 10  end_lat             999260 non-null   float64
 11  end_lng             999260 non-null   float64
 12  member_casual       1002704 non-null  str    
dtypes: float64(4), str(9)
memory usage: 99.5 MB


Ride Duration

In [7]:
# Convert the text columns into actual datetime objects
citibike_df["started_at"] = pd.to_datetime(citibike_df["started_at"])
citibike_df["ended_at"] = pd.to_datetime(citibike_df["ended_at"])

In [8]:
citibike_df["ride_duration_minutes"] = (
    citibike_df["ended_at"] - citibike_df["started_at"]
).dt.total_seconds() / 60

In [9]:
citibike_df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,ride_duration_minutes
0,9F734BE1BFC45FF4,electric_bike,2025-11-18 18:34:14.943,2025-11-18 18:47:33.391,Glenwood Ave,JC094,West Side Ave & Stegman Pkwy,JC131,40.727551,-74.071061,40.710870,-74.093680,member,13.307467
1,B6C773B13AC0E465,classic_bike,2025-11-26 16:29:15.513,2025-11-26 16:43:45.235,Glenwood Ave,JC094,West Side Ave & Stegman Pkwy,JC131,40.727551,-74.071061,40.710870,-74.093680,member,14.495367
2,C300465AA158280F,electric_bike,2025-11-04 22:31:58.010,2025-11-04 22:38:57.017,Bloomfield St & 15 St,HB203,Marshall St & 2 St,HB408,40.754530,-74.026580,40.740802,-74.042521,member,6.983450
3,31A424FC97C8AAFB,classic_bike,2025-11-08 06:51:57.424,2025-11-08 06:57:45.627,Clinton St & 7 St,HB303,Marshall St & 2 St,HB408,40.745420,-74.033320,40.740802,-74.042521,member,5.803383
4,08C5EA04CB1FDC57,classic_bike,2025-11-24 20:31:21.758,2025-11-24 20:38:01.261,Clinton St & 7 St,HB303,Marshall St & 2 St,HB408,40.745420,-74.033320,40.740802,-74.042521,member,6.658383


In [10]:
citibike_df.shape

(1002704, 14)

Droping Missing Values

In [11]:
citibike_df.columns

Index(['ride_id', 'rideable_type', 'started_at', 'ended_at',
       'start_station_name', 'start_station_id', 'end_station_name',
       'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
       'member_casual', 'ride_duration_minutes'],
      dtype='str')

In [12]:
['ride_id', 'started_at', 'ended_at',
       'start_station_name', 'start_station_id', 'end_station_name',
       'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
        ]

['ride_id',
 'started_at',
 'ended_at',
 'start_station_name',
 'start_station_id',
 'end_station_name',
 'end_station_id',
 'start_lat',
 'start_lng',
 'end_lat',
 'end_lng']

In [19]:
# մենակ նշված սյուներինն եմ ջնջել
citibike_df = citibike_df.dropna(
    subset=[
        "end_station_id",
        "end_station_name",
        "start_station_name",
        "start_station_id",
        "ride_id",
        "started_at",
        "ended_at",
        "start_lat",
        "start_lng",
        "end_lat",
        "end_lng",
    ]
)

citibike_df = citibike_df[
    (citibike_df['ride_duration_minutes']>1) & (citibike_df['ride_duration_minutes']<=24*60)
].copy()

In [20]:
missing_values = citibike_df.isnull().sum().reset_index()

missing_values.columns = ['column_name', 'missing_count']

missing_values["missing_percentage"] = (missing_values["missing_count"] / len(citibike_df)) * 100

missing_values.sort_values(by="missing_percentage", ascending=False, inplace=True)

missing_values

,column_name,missing_count,missing_percentage
0,ride_id,0,0.0
1,rideable_type,0,0.0
2,started_at,0,0.0
3,ended_at,0,0.0
4,start_station_name,0,0.0
5,start_station_id,0,0.0
6,end_station_name,0,0.0
7,end_station_id,0,0.0
8,start_lat,0,0.0
9,start_lng,0,0.0


In [21]:
citibike_df[
    [
        'ride_id', 'started_at', 'ended_at',
        'start_station_name',
        'start_station_id', 'end_station_name',
        'end_station_id',
        'start_lat', 'start_lng', 'end_lat', 'end_lng'
    ]
].isna().sum()

ride_id               0
started_at            0
ended_at              0
start_station_name    0
start_station_id      0
end_station_name      0
end_station_id        0
start_lat             0
start_lng             0
end_lat               0
end_lng               0
dtype: int64

Adding Time Based Variables

In [22]:
citibike_df["date"] = citibike_df["started_at"].dt.date
citibike_df["month"] = citibike_df["started_at"].dt.to_period("M").astype(str)
citibike_df["month_name"] = citibike_df["started_at"].dt.month_name()
citibike_df["day_of_week"] = citibike_df["started_at"].dt.day_name()
citibike_df["hour"] = citibike_df["started_at"].dt.hour

In [23]:
citibike_df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,ride_duration_minutes,date,month,month_name,day_of_week,hour
0,9F734BE1BFC45FF4,electric_bike,2025-11-18 18:34:14.943,2025-11-18 18:47:33.391,Glenwood Ave,JC094,West Side Ave & Stegman Pkwy,JC131,40.727551,-74.071061,40.710870,-74.093680,member,13.307467,2025-11-18,2025-11,November,Tuesday,18
1,B6C773B13AC0E465,classic_bike,2025-11-26 16:29:15.513,2025-11-26 16:43:45.235,Glenwood Ave,JC094,West Side Ave & Stegman Pkwy,JC131,40.727551,-74.071061,40.710870,-74.093680,member,14.495367,2025-11-26,2025-11,November,Wednesday,16
2,C300465AA158280F,electric_bike,2025-11-04 22:31:58.010,2025-11-04 22:38:57.017,Bloomfield St & 15 St,HB203,Marshall St & 2 St,HB408,40.754530,-74.026580,40.740802,-74.042521,member,6.983450,2025-11-04,2025-11,November,Tuesday,22
3,31A424FC97C8AAFB,classic_bike,2025-11-08 06:51:57.424,2025-11-08 06:57:45.627,Clinton St & 7 St,HB303,Marshall St & 2 St,HB408,40.745420,-74.033320,40.740802,-74.042521,member,5.803383,2025-11-08,2025-11,November,Saturday,6
4,08C5EA04CB1FDC57,classic_bike,2025-11-24 20:31:21.758,2025-11-24 20:38:01.261,Clinton St & 7 St,HB303,Marshall St & 2 St,HB408,40.745420,-74.033320,40.740802,-74.042521,member,6.658383,2025-11-24,2025-11,November,Monday,20


In [24]:
def assign_season(month_number):
    if month_number in [12, 1, 2]:
        return "Winter"
    elif month_number in [3, 4, 5]:
        return "Spring"
    elif month_number in [6, 7, 8]:
        return "Summer"
    else:
        return "Autumn"


citibike_df["season"] = (
    citibike_df["started_at"]
    .dt.month
    .apply(assign_season)
)

In [25]:
citibike_df[
    [
        "started_at",
        "date",
        "month",
        "month_name",
        "day_of_week",
        "hour",
        "season"
    ]
].head()

,started_at,date,month,month_name,day_of_week,hour,season
0,2025-11-18 18:34:14.943,2025-11-18,2025-11,November,Tuesday,18,Autumn
1,2025-11-26 16:29:15.513,2025-11-26,2025-11,November,Wednesday,16,Autumn
2,2025-11-04 22:31:58.010,2025-11-04,2025-11,November,Tuesday,22,Autumn
3,2025-11-08 06:51:57.424,2025-11-08,2025-11,November,Saturday,6,Autumn
4,2025-11-24 20:31:21.758,2025-11-24,2025-11,November,Monday,20,Autumn


In [26]:
citibike_df.to_csv("../data/JC/JC2025_Enriched.csv", index = False)

In [27]:
citibike_df

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,ride_duration_minutes,date,month,month_name,day_of_week,hour,season
0,9F734BE1BFC45FF4,electric_bike,2025-11-18 18:34:14.943,2025-11-18 18:47:33.391,Glenwood Ave,JC094,West Side Ave & Stegman Pkwy,JC131,40.727551,-74.071061,40.710870,-74.093680,member,13.307467,2025-11-18,2025-11,November,Tuesday,18,Autumn
1,B6C773B13AC0E465,classic_bike,2025-11-26 16:29:15.513,2025-11-26 16:43:45.235,Glenwood Ave,JC094,West Side Ave & Stegman Pkwy,JC131,40.727551,-74.071061,40.710870,-74.093680,member,14.495367,2025-11-26,2025-11,November,Wednesday,16,Autumn
2,C300465AA158280F,electric_bike,2025-11-04 22:31:58.010,2025-11-04 22:38:57.017,Bloomfield St & 15 St,HB203,Marshall St & 2 St,HB408,40.754530,-74.026580,40.740802,-74.042521,member,6.983450,2025-11-04,2025-11,November,Tuesday,22,Autumn
3,31A424FC97C8AAFB,classic_bike,2025-11-08 06:51:57.424,2025-11-08 06:57:45.627,Clinton St & 7 St,HB303,Marshall St & 2 St,HB408,40.745420,-74.033320,40.740802,-74.042521,member,5.803383,2025-11-08,2025-11,November,Saturday,6,Autumn
4,08C5EA04CB1FDC57,classic_bike,2025-11-24 20:31:21.758,2025-11-24 20:38:01.261,Clinton St & 7 St,HB303,Marshall St & 2 St,HB408,40.745420,-74.033320,40.740802,-74.042521,member,6.658383,2025-11-24,2025-11,November,Monday,20,Autumn
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1002699,6E94923FE2C0FF31,electric_bike,2025-08-31 01:07:38.410,2025-08-31 01:10:48.312,Hilltop,JC019,Oakland Ave,JC022,40.731169,-74.057574,40.737604,-74.052478,member,3.165033,2025-08-31,2025-08,August,Sunday,1,Summer
1002700,BBE94D32AA492DA8,electric_bike,2025-08-06 17:14:05.249,2025-08-06 17:26:19.124,River St & Newark St,HB106,Stevens - River Ter & 6 St,HB602,40.736722,-74.029007,40.743133,-74.026989,member,12.231250,2025-08-06,2025-08,August,Wednesday,17,Summer
1002701,68D356CA1B595A75,electric_bike,2025-08-18 00:56:54.543,2025-08-18 01:00:15.680,Stevens - River Ter & 6 St,HB602,Stevens - River Ter & 6 St,HB602,40.743133,-74.026989,40.743133,-74.026989,member,3.352283,2025-08-18,2025-08,August,Monday,0,Summer
1002702,8B55D76CD09DB7E2,electric_bike,2025-08-08 21:14:26.007,2025-08-08 21:22:04.620,Stevens - River Ter & 6 St,HB602,Stevens - River Ter & 6 St,HB602,40.743133,-74.026989,40.743133,-74.026989,member,7.643550,2025-08-08,2025-08,August,Friday,21,Summer
